# EgoVLPv2 Distractor Generation
This notebook sets up the environment, patches the EgoVLPv2 model for compatibility, and generates "hard negative" distractors for EPIC-KITCHENS videos.

## 1. Setup Environment

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Clone EgoVLPv2 repository
!git clone https://github.com/facebookresearch/EgoVLPv2.git
%cd EgoVLPv2

Cloning into 'EgoVLPv2'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 283 (delta 62), reused 46 (delta 46), pack-reused 190 (from 1)
Receiving objects: 100% (283/283), 7.51 MiB | 17.16 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/EgoVLPv2


In [4]:
# 1. Install Python 3.10 and dev headers
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-dev python3.10-distutils libpython3.10-dev -y
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --install /usr/bin/python python /usr/bin/python3.10 1
!curl https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python get-pip.py --force-reinstall
!python --version

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,225 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restrict

In [5]:
!pip install decord einops timm==0.4.12 transformers==4.28.1 ftfy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 123.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 147.3 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [ ]:

!pip install transformers timm decord av einops ffmpeg-python omegaconf


!sed -i 's/from transformers.modeling_utils import (/from transformers.models.roberta.modeling_roberta import (/' /content/EgoVLPv2/EgoVLPv2/model/roberta.py
!sed -i "s/NUM_FUSE_BLOCK = config_yaml\['num_fuse_block'\]/NUM_FUSE_BLOCK = config_yaml.get('num_fuse_block', 4)/" /content/EgoVLPv2/EgoVLPv2/model/video_transformer.py

  Using cached decord-0.6.0-py3-none-manylinux2010_x86_64.whl.metadata (422 bytes)
Using cached decord-0.6.0-py3-none-manylinux2010_x86_64.whl (13.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 46.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [av]


In [12]:
# Download the EK-100 fine-tuned checkpoint
import os

checkpoint_dir = "/content/drive/MyDrive/files_tim/egovlpv2_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(checkpoint_dir, "EK-100_finetune.pth")
if not os.path.exists(checkpoint_path):
    !wget -O "{checkpoint_path}" "http://www.cis.jhu.edu/~shraman/EgoVLPv2/ckpts/EK-100_Finetuned/EK-100_finetune.pth"
else:
    print(f"Checkpoint already exists at {checkpoint_path}")

Checkpoint already exists at /content/drive/MyDrive/files_tim/egovlpv2_checkpoints/EK-100_finetune.pth
Checkpoint size: 3.09 GB


In [ ]:
# Check pretrained checkpoint for cross-attention weights
pretrained_ckpt = torch.load(pretrained_path, map_location='cpu', weights_only=False)
state_dict = pretrained_ckpt['state_dict']

# cross-attention related weights
cross_attn = [k for k in state_dict.keys() if 'cross' in k.lower() or 't2i' in k.lower() or 'alpha' in k.lower()]
print(f"Cross-modal weights: {len(cross_attn)}")
for k in cross_attn[:10]:
    print(f"  {k}")

# projection layers
proj = [k for k in state_dict.keys() if 'proj' in k.lower()]
print(f"\nProjection weights: {len(proj)}")
for k in proj[:10]:
    print(f"  {k}")

In [ ]:
# Check if cross-attention weights were actually loaded
model_state = model.state_dict()
ckpt_state = pretrained_ckpt['state_dict']

# Compare one weight
model_key = 'text_model.encoder.layer.6.alpha_t2i'
ckpt_key = 'module.text_model.encoder.layer.6.alpha_t2i'

print(f"Model key exists: {model_key in model_state}")
print(f"Checkpoint key exists: {ckpt_key in ckpt_state}")

if model_key in model_state and ckpt_key in ckpt_state:
    model_val = model_state[model_key]
    ckpt_val = ckpt_state[ckpt_key]
    print(f"\nModel weight: {model_val}")
    print(f"Checkpoint weight: {ckpt_val}")
    print(f"Match: {torch.allclose(model_val.cpu(), ckpt_val.cpu())}")
else:
    # Check what keys the model actually has
    alpha_keys = [k for k in model_state.keys() if 'alpha' in k]
    print(f"\nModel alpha keys: {alpha_keys}")

## 2. Load EgoVLPv2 Model

In [14]:
!pip install decord

In [15]:
# Download TimeSformer initialization weights
timesformer_init = "/content/EgoVLPv2/EgoVLPv2/pretrained/jx_vit_base_p16_224-80ecf9dd.pth"
os.makedirs(os.path.dirname(timesformer_init), exist_ok=True)
if not os.path.exists(timesformer_init):
    !wget -O {timesformer_init} https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
print(f"TimeSformer weights at: {timesformer_init}")

--2025-12-28 13:30:30--  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth [following]
--2025-12-28 13:30:30--  https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/168799526/65360900-1a09-11eb-8b86-f0a014a6f156?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-28T14%3A05%3A35Z&rscd=attachment%3B+filename%3Djx_vit_base_p16_224-80ecf9dd.pth&rsct=application%2Foctet-stream&skoid=96c2d410

In [ ]:
# Fix the hardcoded TimeSformer path in model.py
model_py_path = '/content/EgoVLPv2/EgoVLPv2/model/model.py'
with open(model_py_path, 'r') as f:
    content = f.read()

# Replace the hardcoded path with our local path
old_path = '/cis/home/shraman/works_meta_2022/pre-training/EgoVLP_Fused_HardNegITM_Checkpoint_multinode/frozen-in-time-main/pretrained/jx_vit_base_p16_224-80ecf9dd.pth'
new_path = '/content/EgoVLPv2/EgoVLPv2/pretrained/jx_vit_base_p16_224-80ecf9dd.pth'

if old_path in content:
    content = content.replace(old_path, new_path)
    with open(model_py_path, 'w') as f:
        f.write(content)
else:
    pass

# Also patch model_epic_charades.py if it exists
model_epic_path = '/content/EgoVLPv2/EgoVLPv2/model/model_epic_charades.py'
if os.path.exists(model_epic_path):
    with open(model_epic_path, 'r') as f:
        content = f.read()
    if old_path in content:
        content = content.replace(old_path, new_path)
        with open(model_epic_path, 'w') as f:
            f.write(content)

In [ ]:
# Create complete config file with all required keys
import yaml

config_path = './EgoNCE_MLM_ITM_Config.yml'

complete_config = {
    'input_image_embed_size': 768,
    'vocab_size': 50265,
    'mlm_prob': 0.15,
    'input_text_embed_size': 768,
    'hidden_size': 768,
    'num_heads': 12,
    'num_layers': 12,
    'mlp_ratio': 4,
    'drop_rate': 0.1,
    'num_fuse_block': 6,  # for video_transformer.py
    'use_checkpoint': True,
    'decay_power': 'cosine',
    'end_lr': 0.0000001,
    'warmup_steps': 0.1
}

with open(config_path, 'w') as f:
    yaml.dump(complete_config, f)

print(f"Config saved with keys: {list(complete_config.keys())}")

Config saved with keys: ['input_image_embed_size', 'vocab_size', 'mlm_prob', 'input_text_embed_size', 'hidden_size', 'num_heads', 'num_layers', 'mlp_ratio', 'drop_rate', 'num_fuse_block', 'use_checkpoint', 'decay_power', 'end_lr', 'warmup_steps']


In [ ]:
# import the model
import sys
sys.path.insert(0, '/content/EgoVLPv2/EgoVLPv2')

import torch
import numpy as np
from PIL import Image
import decord
from transformers import RobertaTokenizerFast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Fix PyTorch 2.6 weights_only issue
import os

# Patch model.py to add weights_only=False
model_path = '/content/EgoVLPv2/EgoVLPv2/model/model.py'
with open(model_path, 'r') as f:
    content = f.read()

# Replace torch.load calls to add weights_only=False
content = content.replace(
    'torch.load(load_checkpoint, map_location=\'cpu\')',
    'torch.load(load_checkpoint, map_location=\'cpu\', weights_only=False)'
)
content = content.replace(
    'torch.load("/content/EgoVLPv2/EgoVLPv2/pretrained/jx_vit_base_p16_224-80ecf9dd.pth", map_location="cpu")',
    'torch.load("/content/EgoVLPv2/EgoVLPv2/pretrained/jx_vit_base_p16_224-80ecf9dd.pth", map_location="cpu", weights_only=False)'
)

with open(model_path, 'w') as f:
    f.write(content)


# reload the module 

import importlib
import sys

# Remove cached module
for mod in list(sys.modules.keys()):
    if 'model' in mod:
        del sys.modules[mod]

# Re-import
from model.model import FrozenInTime

# load the model
import torch

checkpoint_path = "/content/drive/MyDrive/files_tim/egovlpv2_checkpoints/EK-100_finetune.pth"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

video_params = {
    "model": "SpaceTimeTransformer",
    "arch_config": "base_patch16_224",
    "num_frames": 16,
    "pretrained": True,
    "time_init": "zeros",
    "drop_path_rate": 0.0
}

text_params = {
    "model": "roberta-base",
    "pretrained": True,
    "input": "text"
}

model = FrozenInTime(
    video_params=video_params,
    text_params=text_params,
    projection_dim=256,
    projection='minimal',
    load_checkpoint=pretrained_path
)
model = model.to(device)
model.eval()

model = model.to(device)
model.eval()
print("Model loaded successfully!")

Patched torch.load calls with weights_only=False


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Module reloaded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['encoder.layer.10.alpha_t2i', 'encoder.layer.10.crossattention_t2i.output.dense.bias', 'encoder.layer.10.crossattention_t2i.output.dense.weight', 'encoder.layer.10.crossattention_t2i.self.key.bias', 'encoder.layer.10.crossattention_t2i.self.key.weight', 'encoder.layer.10.crossattention_t2i.self.query.bias', 'encoder.layer.10.crossattention_t2i.self.query.weight', 'encoder.layer.10.crossattention_t2i.self.value.bias', 'encoder.layer.10.crossattention_t2i.self.value.weight', 'encoder.layer.11.alpha_t2i', 'encoder.layer.11.crossattention_t2i.output.dense.bias', 'encoder.layer.11.crossattention_t2i.output.dense.weight', 'encoder.layer.11.crossattention_t2i.self.key.bias', 'encoder.layer.11.crossattention_t2i.self.key.weight', 'encoder.layer.11.crossattention_t2i.self.query.bias', 'encoder.layer.11.crossattention_t2i.self.query.weight', 'encoder.layer.11.crossattention_t2i

######USING ATTENTION STYLE:  frozen-in-time
### loaded SpaceTimeTransformer model has FEWER frames than current...### loading weights, filling in the extras via bilinear
Model loaded successfully!


In [20]:
# Initialize tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded!


## 3. Load EPIC-KITCHENS-55 Annotations

In [21]:
import pandas as pd

# Clone annotations if not present
if not os.path.exists('/content/epic-kitchens-55-annotations'):
    !git clone https://github.com/epic-kitchens/epic-kitchens-55-annotations.git /content/epic-kitchens-55-annotations

# Load action segments for P01_01
annotations_path = '/content/epic-kitchens-55-annotations/EPIC_train_action_labels.csv'
df_actions = pd.read_csv(annotations_path)

# Filter for P01_01
df_p01_01 = df_actions[df_actions['video_id'] == 'P01_01'].copy()
print(f"Number of actions in P01_01: {len(df_p01_01)}")
df_p01_01.head()

Cloning into '/content/epic-kitchens-55-annotations'...
remote: Enumerating objects: 675, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 675 (delta 93), reused 62 (delta 43), pack-reused 551 (from 1)
Receiving objects: 100% (675/675), 22.53 MiB | 12.88 MiB/s, done.
Resolving deltas: 100% (449/449), done.
Number of actions in P01_01: 326


,uid,participant_id,video_id,narration,start_timestamp,stop_timestamp,start_frame,stop_frame,verb,verb_class,noun,noun_class,all_nouns,all_noun_classes
0,0,P01,P01_01,open door,00:00:00.14,00:00:03.37,8,202,open,2,door,8,['door'],[8]
1,1,P01,P01_01,turn on light,00:00:04.37,00:00:06.17,262,370,turn-on,12,light,113,['light'],[113]
2,2,P01,P01_01,close door,00:00:06.98,00:00:09.49,418,569,close,3,door,8,['door'],[8]
3,3,P01,P01_01,open fridge,00:00:12.77,00:00:13.99,766,839,open,2,fridge,10,['fridge'],[10]
4,4,P01,P01_01,take celery,00:00:15.25,00:00:16.40,915,983,take,0,celery,185,['celery'],[185]


In [22]:
df_p01_16 = df_actions[df_actions['video_id'] == 'P01_16'].copy()
print(f"Number of actions in P01_16: {len(df_p01_16)}")
df_p01_16.head()

Number of actions in P01_16: 73


,uid,participant_id,video_id,narration,start_timestamp,stop_timestamp,start_frame,stop_frame,verb,verb_class,noun,noun_class,all_nouns,all_noun_classes
1917,2809,P01,P01_16,switch on lights,00:00:00.25,00:00:02.41,15,144,switch-on,12,light,113,['light'],[113]
1918,2810,P01,P01_16,close door,00:00:02.60,00:00:05.59,156,335,close,3,door,8,['door'],[8]
1919,2811,P01,P01_16,take cutting board,00:00:11.33,00:00:14.58,679,874,take,0,board:cutting,19,['board:cutting'],[19]
1920,2812,P01,P01_16,put down cutting board,00:00:13.50,00:00:14.75,810,885,put-down,1,board:cutting,19,['board:cutting'],[19]
1921,2813,P01,P01_16,take knife,00:00:15.20,00:00:16.45,912,987,take,0,knife,5,['knife'],[5]


In [ ]:
import pandas as pd
import os

# 1. Clone annotations if not present
if not os.path.exists('/content/epic-kitchens-55-annotations'):
    !git clone https://github.com/epic-kitchens/epic-kitchens-55-annotations.git /content/epic-kitchens-55-annotations

# 2. Define your specific target videos
target_videos = [
    'P01_16', 'P01_18', 'P02_11', 'P03_04', 'P14_01',
    'P14_02', 'P14_03', 'P14_04', 'P14_05', 'P14_07', 'P14_09'
]

# 3. Load all three potential annotation files

train_path = '/content/epic-kitchens-55-annotations/EPIC_train_action_labels.csv'

df_actions = pd.read_csv(train_path)

# 4. Filter for target list
df_all = df_actions[df_actions['video_id'].isin(target_videos)].copy()

# 5. Check if any videos were missing from the train labels
found_videos = df_all['video_id'].unique()
missing_videos = set(target_videos) - set(found_videos)

print(f"Total actions across all selected videos: {len(df_all)}")
df_all.head()

Total actions across all selected videos: 1240


,uid,participant_id,video_id,narration,start_timestamp,stop_timestamp,start_frame,stop_frame,verb,verb_class,noun,noun_class,all_nouns,all_noun_classes
1917,2809,P01,P01_16,switch on lights,00:00:00.25,00:00:02.41,15,144,switch-on,12,light,113,['light'],[113]
1918,2810,P01,P01_16,close door,00:00:02.60,00:00:05.59,156,335,close,3,door,8,['door'],[8]
1919,2811,P01,P01_16,take cutting board,00:00:11.33,00:00:14.58,679,874,take,0,board:cutting,19,['board:cutting'],[19]
1920,2812,P01,P01_16,put down cutting board,00:00:13.50,00:00:14.75,810,885,put-down,1,board:cutting,19,['board:cutting'],[19]
1921,2813,P01,P01_16,take knife,00:00:15.20,00:00:16.45,912,987,take,0,knife,5,['knife'],[5]


In [23]:
# Load verb and noun class names
df_verbs = pd.read_csv('/content/epic-kitchens-55-annotations/EPIC_verb_classes.csv')
df_nouns = pd.read_csv('/content/epic-kitchens-55-annotations/EPIC_noun_classes.csv')

# Create mappings
verb_id_to_name = dict(zip(df_verbs['verb_id'], df_verbs['class_key']))
noun_id_to_name = dict(zip(df_nouns['noun_id'], df_nouns['class_key']))

print(f"Number of verb classes: {len(verb_id_to_name)}")
print(f"Number of noun classes: {len(noun_id_to_name)}")

Number of verb classes: 125
Number of noun classes: 352


In [24]:
# Generate all possible action descriptions
# For efficiency, we'll use the most common verbs (top 100) and nouns (top 300)

# Get action frequencies
verb_counts = df_actions['verb_class'].value_counts()
noun_counts = df_actions['noun_class'].value_counts()

top_verbs = verb_counts.head(100).index.tolist()
top_nouns = noun_counts.head(300).index.tolist()

print(f"Using top {len(top_verbs)} verbs and {len(top_nouns)} nouns")

# Generate candidate action texts
candidate_actions = []
for verb_id in top_verbs:
    verb_name = verb_id_to_name.get(verb_id, str(verb_id))
    for noun_id in top_nouns:
        noun_name = noun_id_to_name.get(noun_id, str(noun_id))
        # Clean up the action text
        action_text = f"{verb_name} {noun_name}".replace(':', ' ').replace('_', ' ')
        candidate_actions.append({
            'text': action_text,
            'verb_id': verb_id,
            'noun_id': noun_id,
            'verb_name': verb_name,
            'noun_name': noun_name
        })

print(f"Total candidate actions: {len(candidate_actions)}")
print(f"Sample actions: {[a['text'] for a in candidate_actions[:5]]}")

Using top 100 verbs and 300 nouns
Total candidate actions: 30000
Sample actions: ['put tap', 'put plate', 'put cupboard', 'put pan', 'put spoon']


## 4. Prepare Video Data

In [26]:
import tarfile
import os
import glob

# 1. Path to the root folder on Drive and local disk
drive_root_path = "/content/drive/MyDrive/EPIC-KITCHENS55"
local_extract_path = "/content/frames"
os.makedirs(local_extract_path, exist_ok=True)

# 2. THE FILTER LIST: Only these video_ids will be processed
target_videos = [
    'P01_16', 'P01_18', 'P02_11', 'P03_04', 'P14_01',
    'P14_02', 'P14_03', 'P14_04', 'P14_05', 'P14_07', 'P14_09'
]

# 3. Find all participant folders (e.g., P01, P02, etc.)
participant_folders = sorted(glob.glob(os.path.join(drive_root_path, "P*")))

for participant_path in participant_folders:
    participant_id = os.path.basename(participant_path)

    # Find all .tar files within this participant's folder
    tar_files = sorted(glob.glob(os.path.join(participant_path, "*.tar")))

    for tar_file in tar_files:
        file_name = os.path.basename(tar_file)
        video_id = file_name.replace(".tar", "")

        # CHECK 1: Is this video in our target list?
        if video_id not in target_videos:
            continue

        # Define organized local path: /content/frames/P01/P01_16/
        target_dir = os.path.join(local_extract_path, participant_id, video_id)

        # CHECK 2: Does the folder already exist and contain files?
        if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
            print(f"Skipping {video_id} - already extracted at {target_dir}")
            continue

        # If we reached here, proceed with extraction
        os.makedirs(target_dir, exist_ok=True)
        print(f"Extracting {file_name} to {target_dir}...", end=" ")

        try:
            with tarfile.open(tar_file, 'r') as tar:
                tar.extractall(path=target_dir)
            print("Done.")
        except Exception as e:
            print(f"Error extracting {file_name}: {e}")

print(f"\nTarget extractions complete. Files are at: {local_extract_path}")

Skipping P01_16 - already extracted at /content/frames/P01/P01_16
Skipping P01_18 - already extracted at /content/frames/P01/P01_18
Extracting P02_11.tar to /content/frames/P02/P02_11... 

/tmp/ipython-input-2642090420.py:47: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=target_dir)


Done.
Extracting P03_04.tar to /content/frames/P03/P03_04... Done.
Extracting P14_01.tar to /content/frames/P14/P14_01... Done.
Extracting P14_02.tar to /content/frames/P14/P14_02... Done.
Extracting P14_03.tar to /content/frames/P14/P14_03... Done.
Extracting P14_04.tar to /content/frames/P14/P14_04... Done.
Extracting P14_05.tar to /content/frames/P14/P14_05... Done.
Extracting P14_07.tar to /content/frames/P14/P14_07... Done.
Extracting P14_09.tar to /content/frames/P14/P14_09... Done.

Target extractions complete. Files are at: /content/frames


In [27]:
import os

# Update these to match your actual Drive locations
frames_dir = "/content/frames"

if os.path.exists(frames_dir):
    print(f"Frames found at: {frames_dir}")
    use_video = False

Frames found at: /content/frames


In [28]:
import torchvision.transforms as transforms
import torchvision.transforms._transforms_video as transforms_video

# Transforms for video preprocessing
def get_transforms(input_res=224):
    return transforms.Compose([
        transforms.Resize(input_res),
        transforms.CenterCrop(input_res),
        transforms_video.NormalizeVideo(
            mean=[123.675, 116.28, 103.53],
            std=[58.395, 57.12, 57.375]
        ),
    ])

video_transforms = get_transforms(224)

def get_frame_ids(start_frame, end_frame, num_frames=16):
    """Get uniformly spaced frame indices"""
    if end_frame <= start_frame:
        end_frame = start_frame + num_frames
    seg_size = (end_frame - start_frame - 1) / num_frames
    frame_ids = []
    for i in range(num_frames):
        start = int(np.round(seg_size * i) + start_frame)
        end = int(np.round(seg_size * (i + 1)) + start_frame)
        frame_id = (start + end) // 2
        frame_ids.append(min(frame_id, end_frame - 1))
    return frame_ids

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


In [ ]:
def load_frames_clip(video_id, participant_id, start_frame, end_frame, num_frames=16):
    """Dynamically locates and loads frames based on participant and video IDs"""

    # Construct the path: /content/frames/P01/P01_16
    specific_video_dir = os.path.join(local_base_path, participant_id, video_id)

    if not os.path.exists(specific_video_dir):
        nested_dir = os.path.join(specific_video_dir, video_id)
        if os.path.exists(nested_dir):
            specific_video_dir = nested_dir
        else:
            raise FileNotFoundError(f"Frame directory not found: {specific_video_dir}")

    frame_ids = get_frame_ids(start_frame, end_frame, num_frames)
    frames = []

    for fid in frame_ids:
        frame_path = os.path.join(specific_video_dir, f"frame_{fid:010d}.jpg")

        if os.path.exists(frame_path):
            img = Image.open(frame_path).convert('RGB')
            frames.append(np.array(img))
        else:
            if frames:
                frames.append(frames[-1].copy())
            else:
                frames.append(np.zeros((224, 224, 3), dtype=np.uint8))

    frames = np.stack(frames, axis=0)
    frames = torch.tensor(frames, dtype=torch.float32).permute(0, 3, 1, 2)

    frames = torch.nn.functional.interpolate(frames, size=(224, 224), mode='bilinear', align_corners=False)

    mean = torch.tensor([123.675, 116.28, 103.53]).view(1, 3, 1, 1)
    std = torch.tensor([58.395, 57.12, 57.375]).view(1, 3, 1, 1)
    frames = (frames - mean) / std
    return frames

## 5. Encode All Candidate Texts

In [31]:
# Pre-encode all candidate action texts for efficiency
@torch.no_grad()
def encode_texts(texts, batch_size=256):
    """Encode all texts in batches"""
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # Tokenize
        tokens = tokenizer(
            batch_texts,
            padding='max_length',
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )

        text_data = {
            'input_ids': tokens['input_ids'].to(device),
            'attention_mask': tokens['attention_mask'].to(device)
        }

        # Get embeddings
        text_emb = model.compute_text(text_data)
        all_embeddings.append(text_emb.cpu())

        if (i // batch_size) % 10 == 0:
            print(f"Encoded {min(i+batch_size, len(texts))}/{len(texts)} texts")

    return torch.cat(all_embeddings, dim=0)

# Encode all candidate actions
print("Encoding candidate action texts...")
candidate_texts = [a['text'] for a in candidate_actions]
text_embeddings = encode_texts(candidate_texts)
print(f"Text embeddings shape: {text_embeddings.shape}")

Encoding candidate action texts...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1621: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Encoded 256/30000 texts
Encoded 2816/30000 texts
Encoded 5376/30000 texts
Encoded 7936/30000 texts
Encoded 10496/30000 texts
Encoded 13056/30000 texts
Encoded 15616/30000 texts
Encoded 18176/30000 texts
Encoded 20736/30000 texts
Encoded 23296/30000 texts
Encoded 25856/30000 texts
Encoded 28416/30000 texts
Text embeddings shape: torch.Size([30000, 256])


## 6. Generate Distractors

In [32]:
@torch.no_grad()
def generate_distractors_for_action(row, text_embeddings, candidate_actions, num_distractors=19):
    """Generate distractors using dynamic pathing"""
    video_id = row['video_id']
    participant_id = row['participant_id']

    try:
        # Call the updated loader with ID info
        video_tensor = load_frames_clip(
            video_id,
            participant_id,
            int(row['start_frame']),
            int(row['stop_frame'])
        )
        video_tensor = video_tensor.unsqueeze(0).to(device) # [1, T, C, H, W]
    except Exception as e:
        print(f"Error loading {video_id} (Action {row['uid']}): {e}")
        return [], None

    # Compute Embeddings and Similarities
    video_emb = model.compute_video(video_tensor)
    video_emb_norm = video_emb / video_emb.norm(dim=-1, keepdim=True)
    text_emb_norm = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
    similarities = (video_emb_norm.cpu() @ text_emb_norm.T).squeeze(0)


    # Get ground truth info
    gt_verb, gt_noun = row['verb_class'], row['noun_class']
    sorted_indices = similarities.argsort(descending=True)
    distractors = []
    for idx in sorted_indices.tolist():
        action = candidate_actions[idx]
        if action['verb_id'] == gt_verb and action['noun_id'] == gt_noun:
            continue
        distractors.append({'answer': action['text'], 'confidence': f"{similarities[idx].item():.4f}"})
        if len(distractors) >= num_distractors: break

    return distractors, similarities

In [ ]:
# The base path where you extracted the 11 target video tars
local_base_path = "/content/frames"

# The 11 videos you actually extracted
target_videos = [
    'P01_16', 'P01_18', 'P02_11', 'P03_04', 'P14_01',
    'P14_02', 'P14_03', 'P14_04', 'P14_05', 'P14_07', 'P14_09'
]

# Create a clean dataframe containing only these 11 videos
df_all = df_actions[df_actions['video_id'].isin(target_videos)].copy().reset_index(drop=True)

print(f"Cleaned DataFrame: {len(df_all)} actions to process across {len(target_videos)} videos.")

Cleaned DataFrame: 1240 actions to process across 11 videos.


In [ ]:
# Generate distractors for all actions

results = []
for idx, row in df_all.iterrows():
    print(f"[{idx+1}/{len(df_all)}] Processing {row['video_id']} - {row['narration']}")

    distractors, sims = generate_distractors_for_action(row, text_embeddings, candidate_actions)

    if not distractors:
        continue

    # Get ground truth
    gt_verb_name = verb_id_to_name.get(row['verb_class'], str(row['verb_class']))
    gt_noun_name = noun_id_to_name.get(row['noun_class'], str(row['noun_class']))

    # Build frame indices
    start_frame = int(row['start_frame'])
    stop_frame = int(row['stop_frame'])
    n_frames = 16
    frame_indices = get_frame_ids(start_frame, stop_frame, n_frames)

    # Format distractors: only answer and confidence
    formatted_distractors = [
        {"answer": d['answer'], "confidence": d['confidence']}
        for d in distractors
    ]

    result = {
        "uid": int(row['uid']),
        "participant_id": row['participant_id'],
        "video_id": row['video_id'],
        "start_timestamp": row['start_timestamp'],
        "stop_timestamp": row['stop_timestamp'],
        "start_frame": start_frame,
        "stop_frame": stop_frame,
        "n_frames": n_frames,
        "frame_indices": frame_indices,
        "ground_truth": {
            "verb": int(row['verb_class']),
            "verb_class": gt_verb_name,
            "noun": int(row['noun_class']),
            "noun_class": gt_noun_name,
            "narration": row['narration']
        },
        "distractors_with_confidence": formatted_distractors,
        "num_options": len(formatted_distractors) + 1  # distractors + ground truth
    }
    results.append(result)

print(f"\nGenerated distractors for {len(results)} actions")

# Save results
output_path = "/content/drive/MyDrive/files_tim/P01_16_EgoVLPv2_distractors.json"
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} action distractors to {output_path}")

[1/1240] Processing P01_16 - switch on lights
[2/1240] Processing P01_16 - close door
[3/1240] Processing P01_16 - take cutting board
[4/1240] Processing P01_16 - put down cutting board
[5/1240] Processing P01_16 - take knife
[6/1240] Processing P01_16 - put down knife
[7/1240] Processing P01_16 - take glass
[8/1240] Processing P01_16 - take glass
[9/1240] Processing P01_16 - open cupboard
[10/1240] Processing P01_16 - open cupboard
[11/1240] Processing P01_16 - put glass into cupboard
[12/1240] Processing P01_16 - put glass into cupboard
[13/1240] Processing P01_16 - take saucepan
[14/1240] Processing P01_16 - open cupboard
[15/1240] Processing P01_16 - take lid
[16/1240] Processing P01_16 - put saucepan into cupboard
[17/1240] Processing P01_16 - put down lid
[18/1240] Processing P01_16 - take colander
[19/1240] Processing P01_16 - put down colander
[20/1240] Processing P01_16 - take container
[21/1240] Processing P01_16 - take lead
[22/1240] Processing P01_16 - put down container an

NameError: name 'json' is not defined

In [40]:
import json
output_path = "/content/drive/MyDrive/files_tim/test_EgoVLPv2_distractors.json"
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} action distractors to {output_path}")

Saved 1240 action distractors to /content/drive/MyDrive/files_tim/test_EgoVLPv2_distractors.json
